# RHI Live Runtime v5
## Contract-Anchored Consensus Gate

This fixes the v4 false-consensus risk.

v4 added the VOTE (Maj) primitive to KRRB: when the top two branches are both
high-quality and their margin is small, return `consensus_collapse` instead of Ω.
That was the right structural addition.

But the agreement check used raw lexical Jaccard — and NEXUS-domain answers share
a dense vocabulary (`contract`, `boundary`, `domain_carrier`, `preserved_function`,
etc.) regardless of whether they stake the same *operational* position.  Two branches
could disagree on *what the domain carrier is* while scoring ≥ 0.36 Jaccard, and v4
would collapse them falsely.

v5 adds the third gate:

```text
prompt
  ↓
slot_builder_lora_v2 emits raw contract
  ↓
contract repair gate fixes polarity and scar leakage
  ↓
base model generates answer branches
  ↓
deterministic five-dimensional critic scores branches
  ↓
KRRB v5 checks:
      normal collapse            (margin >= MARGIN_MIN, both high)
      or consensus gate:
            gate 1 — both high quality
            gate 2 — lexical or audit agreement
            gate 3 — contract-anchored stance agreement   ← NEW
      or Ω_divergent_consensus   (gates 1+2 pass, gate 3 fails) ← NEW
      or Ω
```

Put this notebook in **Downloads**, beside:

```text
slot_builder_lora_v2/
```

Output:

```text
rhi_live_runtime_v5_outputs/
  rhi_live_runs_v5.jsonl
  rhi_live_runtime_v5_manifest.json
```


In [1]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

MODEL_NAME       = "Qwen/Qwen2.5-1.5B-Instruct"
SLOT_ADAPTER_DIR = ROOT / "slot_builder_lora_v2"

OUTPUT_DIR = ROOT / "rhi_live_runtime_v5_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILES_ONLY = True
USE_4BIT         = False

CONTRACT_MAX_NEW_TOKENS = 700
ANSWER_MAX_NEW_TOKENS   = 900
RESIDUE_MAX_NEW_TOKENS  = 500

N_BRANCHES             = 5
DO_SAMPLE_FOR_BRANCHES = True
BRANCH_TEMPERATURE     = 0.55
BRANCH_TOP_P           = 0.92

# v3 deterministic collapse thresholds.
SUPPORT_MIN = 4
MARGIN_MIN  = 0.04
PSI_MIN     = 0.52
AUDIT_MIN   = 0.52

# v4 consensus gate — both-high + agreement test.
CONSENSUS_MARGIN_MAX    = 0.04
CONSENSUS_AGREEMENT_MIN = 0.36   # lexical Jaccard
CONSENSUS_AUDIT_MIN     = 0.86   # audit-vector agreement

# v5 contract-anchored stance gate.
# Minimum per-field stance consistency score to pass the third gate.
CONSENSUS_STANCE_MIN = 0.50

# If true, base model writes residue commentary; numeric scores stay deterministic.
ADD_MODEL_RESIDUE_COMMENTARY = True

print("ROOT:", ROOT)
print("SLOT_ADAPTER_DIR:", SLOT_ADAPTER_DIR, SLOT_ADAPTER_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


ROOT: D:\@User Data\Downloads
SLOT_ADAPTER_DIR: D:\@User Data\Downloads\slot_builder_lora_v2 True
OUTPUT_DIR: D:\@User Data\Downloads\rhi_live_runtime_v5_outputs


In [2]:
# ============================================================
# IMPORTS
# ============================================================
INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "pandas"
    ])

import json
import re
import time
import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
vram GB: 8.0


In [3]:
# ============================================================
# HELPERS
# ============================================================
REQUIRED_FIELDS = [
    "family_class",
    "domain_carrier",
    "forbidden_neighbor_carrier",
    "boundary_conditions",
    "preserved_function",
    "failure_modes",
    "witness_readout",
    "residue",
]

AUDIT_FIELDS = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse"]

STOPWORDS = {
    "the","a","an","and","or","of","to","in","on","for","with","as","is","are",
    "was","were","be","being","been","it","its","this","that","these","those",
    "by","from","into","at","while","what","when","where","why","how","which",
    "who","whom","one","two","three","do","does","did","not","no","yes","can",
    "could","should","would","will","may","might","using","use","used","uses",
    "current","answer","task","prompt","question","response",
}

# High-frequency NEXUS-domain terms that appear in almost every framework answer.
# Used by the stance gate to detect vocabulary inflation vs. real operational agreement.
NEXUS_VOCAB = {
    "contract","boundary","domain_carrier","forbidden","preserved_function","witness_readout",
    "boundary_conditions","failure_modes","family_class","residue","carrier","collapse",
    "nexus","slot","operational","fit","missing","shape","cavity","rhi","krrb",
}

SCAR_TERMS = {
    "string wraps but does not center under rotation",
    "last paragraph is here",
    "unrelated_to_prompt",
    "wrong neighboring domain carrier",
    "surface label without operational fit",
    "general purpose","purpose","not answered","nonexistent",
    "name-only rubber-part match",
    "permanent membership without local constraint satisfaction",
}

GENERIC_DOMAIN_TERMS = {
    "current","failing","form","use","using","properly","understanding",
    "answer","task","prompt","question","response","chosen","collapse",
}

def now_iso():
    return datetime.now().isoformat(timespec="seconds")

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def clamp01(x):
    try:
        return max(0.0, min(1.0, float(x)))
    except Exception:
        return 0.0

def wordset(text: Any) -> set:
    if isinstance(text, list):
        text = " ".join(map(str, text))
    toks = re.findall(r"[a-zA-Z0-9_]+", str(text).lower())
    return {t for t in toks if t not in STOPWORDS and len(t) > 1}

def wordset_no_nexus(text: Any) -> set:
    """wordset minus shared NEXUS framework vocabulary.
    Used by the stance gate so framework boilerplate does not inflate agreement."""
    return wordset(text) - NEXUS_VOCAB

def phrase_present(text: str, phrase: str) -> bool:
    return phrase.lower() in str(text).lower()

def extract_first_json_object(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    text = str(text).strip()
    try:
        obj = json.loads(text)
        return (obj, None) if isinstance(obj, dict) else (None, "json_not_dict")
    except Exception:
        pass
    cleaned = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        obj = json.loads(cleaned)
        return (obj, None) if isinstance(obj, dict) else (None, "fenced_json_not_dict")
    except Exception:
        pass
    start = text.find("{")
    end   = text.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            return (obj, None) if isinstance(obj, dict) else (None, "scanned_json_not_dict")
        except Exception as e:
            return None, "json_parse_error: " + str(e)
    return None, "no_json_object_found"

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("{") or ":" in s:
            return [s]
        return [p.strip() for p in re.split(r"[|,;]", s) if p.strip()]
    return [str(x).strip()]

def normalize_contract(c0: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    c0 = c0 or {}
    return {
        "family_class":               str(c0.get("family_class","") or "").strip(),
        "domain_carrier":             normalize_list(c0.get("domain_carrier",[])),
        "forbidden_neighbor_carrier": normalize_list(c0.get("forbidden_neighbor_carrier",[])),
        "boundary_conditions":        normalize_list(c0.get("boundary_conditions",[])),
        "preserved_function":         str(c0.get("preserved_function","") or "").strip(),
        "failure_modes":              normalize_list(c0.get("failure_modes",[])),
        "witness_readout":            str(c0.get("witness_readout","") or "").strip(),
        "residue":                    c0.get("residue", None),
    }

def contract_complete(c: Optional[Dict[str, Any]]) -> bool:
    if not isinstance(c, dict):
        return False
    c = normalize_contract(c)
    for f in REQUIRED_FIELDS:
        if f == "residue":
            continue
        v = c.get(f)
        if isinstance(v, list) and len(v) == 0:
            return False
        if not isinstance(v, list) and not str(v or "").strip():
            return False
    return True

def stable_dedupe(items):
    out, seen = [], set()
    for x in items:
        s = str(x).strip()
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    return out

# ── v5: contract-anchored stance gate ────────────────────────────────────────

def _field_words(contract: Dict[str, Any], field: str) -> set:
    """Extract non-NEXUS-inflated word set from one contract field."""
    val = contract.get(field, [])
    if isinstance(val, list):
        combined = " ".join(val)
    else:
        combined = str(val or "")
    return wordset_no_nexus(combined)

def contract_stance_agreement(
    contract: Dict[str, Any],
    answer_a: str,
    answer_b: str,
) -> Dict[str, Any]:
    """
    Check whether answer_a and answer_b take the same operational stance
    on each contract-grounded dimension.

    For every contract field that carries signal (non-empty after NEXUS-vocab
    removal), we measure how much each answer overlaps with that field's terms.
    Stance consistency on a field = 1 - |overlap_a - overlap_b|.

    A field with no signal contributes neutral (1.0) so it does not penalise.
    Aggregate = mean over checked fields.

    Returns:
        {
          "aggregate": float,      # mean stance consistency across fields
          "fields": {field: {
              "words":     list,   # contract terms driving the check
              "overlap_a": float,
              "overlap_b": float,
              "consistency": float,
          }},
          "passed": bool,          # aggregate >= CONSENSUS_STANCE_MIN
        }
    """
    c   = normalize_contract(contract)
    wa  = wordset_no_nexus(answer_a)
    wb  = wordset_no_nexus(answer_b)

    STANCE_FIELDS = [
        "domain_carrier",
        "preserved_function",
        "boundary_conditions",
        "family_class",
        "witness_readout",
    ]

    results = {}
    for field in STANCE_FIELDS:
        fw = _field_words(c, field)
        if not fw:
            results[field] = {
                "words": [], "overlap_a": 1.0, "overlap_b": 1.0, "consistency": 1.0
            }
            continue
        ov_a = safe_div(len(wa & fw), len(fw))
        ov_b = safe_div(len(wb & fw), len(fw))
        consistency = 1.0 - abs(ov_a - ov_b)
        results[field] = {
            "words":       sorted(fw),
            "overlap_a":   round(ov_a, 4),
            "overlap_b":   round(ov_b, 4),
            "consistency": round(consistency, 4),
        }

    aggregate = sum(v["consistency"] for v in results.values()) / len(results)
    return {
        "aggregate": round(aggregate, 4),
        "fields":    results,
        "passed":    aggregate >= CONSENSUS_STANCE_MIN,
    }

print("helpers + stance gate ready")


helpers + stance gate ready


In [4]:
# ============================================================
# LOAD MODEL + SLOT ADAPTER
# ============================================================
if not SLOT_ADAPTER_DIR.exists():
    raise FileNotFoundError("Missing slot adapter folder: " + str(SLOT_ADAPTER_DIR))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, use_fast=True, local_files_only=LOCAL_FILES_ONLY,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
if not USE_4BIT and torch.cuda.is_available():
    base_model = base_model.to("cuda")

model = PeftModel.from_pretrained(
    base_model, SLOT_ADAPTER_DIR, local_files_only=LOCAL_FILES_ONLY,
)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

print("loaded base:", MODEL_NAME)
print("loaded slot adapter:", SLOT_ADAPTER_DIR)
print("device:", device)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W0504 07:03:04.742000 24424 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


loaded base: Qwen/Qwen2.5-1.5B-Instruct
loaded slot adapter: D:\@User Data\Downloads\slot_builder_lora_v2
device: cuda


In [5]:
# ============================================================
# GENERATION
# ============================================================
def render_chat(messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt,
        )
    nl  = chr(10)
    out = [m.get("role","user").upper() + ":" + nl + m.get("content","") for m in messages]
    if add_generation_prompt:
        out.append("ASSISTANT:" + nl)
    return (nl + nl).join(out)

def generate_text(
    messages: List[Dict[str, str]],
    max_new_tokens: int,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    use_slot_adapter: bool = True,
) -> str:
    prompt_text = render_chat(messages, add_generation_prompt=True)
    inputs      = tokenizer(prompt_text, return_tensors="pt").to(device)
    gen_kwargs  = {
        "max_new_tokens": max_new_tokens,
        "do_sample":      do_sample,
        "pad_token_id":   tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"]       = top_p
    with torch.no_grad():
        if use_slot_adapter:
            output_ids = model.generate(**inputs, **gen_kwargs)
        else:
            try:
                with model.disable_adapter():
                    output_ids = model.generate(**inputs, **gen_kwargs)
            except Exception:
                output_ids = model.generate(**inputs, **gen_kwargs)
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("generation ready")


generation ready


In [6]:
# ============================================================
# SLOT BUILDER + CONTRACT REPAIR GATE
# ============================================================
SLOT_SYSTEM = "\n".join([
    "You are the Nexus Slot Constructor.",
    "Generate the missing-shape contract before answer selection.",
    "Do not answer the task.",
    "Do not mention answer choices.",
    "Return strict JSON only.",
    "Required fields: family_class, domain_carrier, forbidden_neighbor_carrier, boundary_conditions, preserved_function, failure_modes, witness_readout, residue.",
    "Use operational fit, not labels.",
])

def build_slot_user_prompt(prompt: str) -> str:
    return "\n".join([
        "Prompt:", prompt, "",
        "Generate the missing-shape contract.",
        "Checklist:",
        "1. Need: occupy the inverse cavity.",
        "2. Function: preserve or redirect the required operation.",
        "3. Boundary: respect constraints.",
        "4. Trap: reject noun/surface-label confusion.",
        "5. Collapse: produce one executable witness/readout.",
        "Return JSON only.",
    ])

def detect_missing_prerequisites(prompt: str) -> Dict[str, Any]:
    p = prompt.lower()
    prereq = {
        "has_before_contract":          bool(re.search(r"before\s+(forming\s+)?a?\s*contract|before\s+contract", p)),
        "has_tool_before_contract":     ("tool" in p or "tools" in p) and ("before" in p and "contract" in p),
        "has_retrieval_before_intent":  ("retrieval" in p or "rag" in p) and ("before" in p and "intent" in p),
    }
    required, forbidden = [], []
    if prereq["has_tool_before_contract"]:
        required.extend(["contract","intent","boundary","sequence","tool use after contract"])
        forbidden.extend(["tool before contract","action without boundary","answer guessing","tool-first action"])
    if prereq["has_retrieval_before_intent"]:
        required.extend(["intent","retrieval","query contract","stabilized need"])
        forbidden.extend(["retrieval before intent","context before contract","semantic drift"])
    prereq["required_terms"]  = stable_dedupe(required)
    prereq["forbidden_terms"] = stable_dedupe(forbidden)
    return prereq

def repair_contract_polarity(contract: Dict[str, Any], prompt: str) -> Tuple[Dict[str, Any], List[str]]:
    c = normalize_contract(contract)
    prereq = detect_missing_prerequisites(prompt)
    repairs = []

    def clean(xs, remove_generic=False):
        out = []
        for x in xs:
            s, low = str(x).strip(), str(x).strip().lower()
            if not s or low in SCAR_TERMS:
                repairs.append("removed_scar:" + s)
                continue
            if remove_generic and low in GENERIC_DOMAIN_TERMS:
                repairs.append("removed_generic_domain:" + s)
                continue
            out.append(s)
        return stable_dedupe(out)

    c["domain_carrier"]             = clean(c["domain_carrier"], remove_generic=True)
    c["forbidden_neighbor_carrier"] = clean(c["forbidden_neighbor_carrier"])
    c["failure_modes"]              = clean(c["failure_modes"])

    if prereq["has_tool_before_contract"]:
        for term in ["contract","intent","boundary","tool","agent","sequence","tool use after contract"]:
            if term not in c["domain_carrier"]:
                c["domain_carrier"].append(term)
                repairs.append("added_positive:" + term)
        new_forbidden = []
        for term in c["forbidden_neighbor_carrier"]:
            if term.lower() in {"contract","forming contract","forming","intent","boundary"}:
                repairs.append("removed_polarity_inverted_forbidden:" + term)
                continue
            new_forbidden.append(term)
        c["forbidden_neighbor_carrier"] = new_forbidden
        for term in prereq["forbidden_terms"]:
            if term not in c["forbidden_neighbor_carrier"]:
                c["forbidden_neighbor_carrier"].append(term)
                repairs.append("added_forbidden:" + term)
        c["preserved_function"]  = "form a contract before tool use so action is gated by intent, boundary, and operational fit"
        c["boundary_conditions"] = [
            "contract must be formed before tool use",
            "tool action must be gated by intent and boundary",
            "reject tool-first action, answer guessing, and surface-label routing",
        ]
        c["witness_readout"] = "the answer explains that agents fail when action/tool use occurs before intent, boundary, and contract are stabilized"
        repairs.append("rewrote_boundary_polarity")

    if len(c["domain_carrier"]) < 5:
        for w in sorted(wordset(prompt)):
            if w not in c["domain_carrier"]:
                c["domain_carrier"].append(w)
                repairs.append("added_prompt_carrier:" + w)
            if len(c["domain_carrier"]) >= 7:
                break

    c["domain_carrier"]             = stable_dedupe(c["domain_carrier"])
    c["forbidden_neighbor_carrier"] = stable_dedupe(c["forbidden_neighbor_carrier"])
    c["failure_modes"]              = stable_dedupe(c["failure_modes"])
    return c, repairs

def generate_contract(prompt: str) -> Dict[str, Any]:
    raw = generate_text(
        [
            {"role": "system", "content": SLOT_SYSTEM},
            {"role": "user",   "content": build_slot_user_prompt(prompt)},
        ],
        max_new_tokens=CONTRACT_MAX_NEW_TOKENS,
        do_sample=False,
        use_slot_adapter=True,
    )
    obj, err = extract_first_json_object(raw)
    original = normalize_contract(obj) if obj else None
    repaired, repairs = repair_contract_polarity(original, prompt) if original else (None, ["parse_failed"])
    return {
        "raw_contract":      raw,
        "contract_original": original,
        "contract":          repaired,
        "parse_error":       err,
        "complete":          contract_complete(repaired),
        "repairs":           repairs,
    }

print("slot builder + contract repair gate ready")


slot builder + contract repair gate ready


In [7]:
# ============================================================
# ANSWER BRANCHES
# ============================================================
BRANCH_SYSTEM = "\n".join([
    "You are an answer generator inside an RHI runtime.",
    "Use the provided contract as the operational target.",
    "Answer the user's prompt directly.",
    "Do not output JSON unless the user asked for JSON.",
    "Do not mention internal scoring.",
    "Be precise. Do not invent facts.",
])

BRANCH_STYLES = [
    ("direct",         "Answer directly with the clearest useful response.", False, 0.0),
    ("operational",    "Answer by identifying operation, boundary, trap, and witness.", False, 0.0),
    ("contract_fit",   "Answer through the contract: need, preserved function, boundary, and witness.", False, 0.0),
    ("skeptical",      "Reject surface-label traps and explain the failure mode before giving the answer.", True, BRANCH_TEMPERATURE),
    ("residue_aware",  "Answer and explicitly flag remaining residue if the contract is incomplete.", True, BRANCH_TEMPERATURE),
]

def branch_user_prompt(prompt: str, contract: Dict[str, Any], instruction: str) -> str:
    return (
        "User prompt:\n" + prompt
        + "\n\nMissing-shape contract:\n" + json.dumps(normalize_contract(contract), ensure_ascii=False, indent=2)
        + "\n\nBranch instruction:\n" + instruction
        + "\n\nReturn the answer only."
    )

def generate_candidate_branches(prompt: str, contract: Dict[str, Any]) -> List[Dict[str, Any]]:
    branches = []
    for name, instruction, sample, temp in BRANCH_STYLES[:N_BRANCHES]:
        text = generate_text(
            [
                {"role": "system", "content": BRANCH_SYSTEM},
                {"role": "user",   "content": branch_user_prompt(prompt, contract, instruction)},
            ],
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
            do_sample=bool(sample and DO_SAMPLE_FOR_BRANCHES),
            temperature=float(temp),
            top_p=BRANCH_TOP_P,
            use_slot_adapter=False,
        )
        branches.append({"branch": name, "instruction": instruction, "answer": text})
    return branches

print("branches ready")


branches ready


In [8]:
# ============================================================
# DETERMINISTIC FIVE-DIMENSIONAL CRITIC
# ============================================================
def overlap_score(answer_text: str, terms: Any) -> Dict[str, Any]:
    a = wordset(answer_text)
    t = wordset(terms)
    hits = sorted(a.intersection(t))
    return {"score": safe_div(len(hits), len(t)), "hits": hits, "n_terms": len(t), "n_hits": len(hits)}

def answer_quality_proxy(answer_text: str) -> float:
    words = re.findall(r"[a-zA-Z0-9_]+", str(answer_text))
    if not words:        return 0.0
    if len(words) < 25:  return 0.25
    if len(words) > 260: return 0.72
    return min(1.0, len(words) / 120.0)

def contains_any(text: str, terms: List[str]) -> bool:
    low = str(text).lower()
    return any(t.lower() in low for t in terms)

def deterministic_operational_audit(prompt: str, contract: Dict[str, Any], answer: str) -> Dict[str, Any]:
    c      = normalize_contract(contract)
    prereq = detect_missing_prerequisites(prompt)

    domain    = overlap_score(answer, c["domain_carrier"])
    function  = overlap_score(answer, c["preserved_function"])
    witness   = overlap_score(answer, c["witness_readout"])
    boundary  = overlap_score(answer, c["boundary_conditions"])
    forbidden = overlap_score(answer, c["forbidden_neighbor_carrier"])
    prompt_fit= overlap_score(answer, prompt)
    quality   = answer_quality_proxy(answer)

    # F_need
    need_signals = sum([
        domain["score"] >= 0.25,
        contains_any(answer, ["before","prior","first","precondition","prerequisite"]),
        contains_any(answer, ["contract","intent","boundary","constraint"]),
        contains_any(answer, ["tool","tools","action","agent"]),
    ])
    F_need = need_signals / 4

    # F_function
    function_signals = sum([
        function["score"] >= 0.20,
        contains_any(answer, ["gate","gated","govern","constrain","stabilize","stabilized"]),
        contains_any(answer, ["intent","need","missing shape","contract"]),
        contains_any(answer, ["tool use","tool call","action","act"]),
        contains_any(answer, ["operational fit","fit","boundary","interface"]),
    ])
    F_function = function_signals / 5

    # F_boundary
    boundary_signals = sum([
        boundary["score"] >= 0.15,
        contains_any(answer, ["before","prior","order","sequence","precondition"]),
        contains_any(answer, ["reject","prevent","block","refuse","gate"]),
        contains_any(answer, ["boundary","constraint","condition","rule"]),
        contains_any(answer, ["tool","action","retrieval","agent"]),
    ])
    F_boundary = boundary_signals / 5

    # F_trap
    trap_signals = sum([
        not contains_any(answer, ["label","name","surface","terminology","vocabulary"]) or
            contains_any(answer, ["trap","confuse","mistake","mismatch"]),
        not (forbidden["score"] > 0.55),
        contains_any(answer, ["fail","failure","problem","error","issue","wrong","incorrect"]),
        contains_any(answer, ["instead","should","must","requires","correct"]),
    ])
    F_trap = trap_signals / 4

    # F_collapse
    collapse_signals = sum([
        witness["score"] >= 0.15,
        contains_any(answer, ["because","therefore","result","so","thus","hence","leads"]),
        contains_any(answer, ["contract","boundary","intent","slot"]),
        contains_any(answer, ["agent","system","model","runtime"]),
    ])
    F_collapse = collapse_signals / 4

    residue = []
    if F_need     < 0.50: residue.append("F_need_weak")
    if F_function < 0.50: residue.append("F_function_weak")
    if F_boundary < 0.50: residue.append("F_boundary_weak")
    if F_trap     < 0.50: residue.append("F_trap_weak")
    if F_collapse < 0.50: residue.append("F_collapse_weak")

    audit = {
        "F_need": round(F_need, 4), "F_function": round(F_function, 4),
        "F_boundary": round(F_boundary, 4), "F_trap": round(F_trap, 4),
        "F_collapse": round(F_collapse, 4), "residue": residue,
    }
    evidence = {
        "domain": domain, "function": function, "witness": witness,
        "boundary": boundary, "forbidden": forbidden, "prompt_fit": prompt_fit,
        "quality": quality,
    }
    return {"audit": audit, "evidence": evidence}

def audit_score(audit: Dict[str, Any]) -> float:
    return float(
        0.24 * audit["F_need"]
        + 0.24 * audit["F_function"]
        + 0.18 * audit["F_boundary"]
        + 0.18 * audit["F_trap"]
        + 0.16 * audit["F_collapse"]
    )

def model_residue_commentary(prompt: str, contract: Dict[str, Any], answer: str, audit: Dict[str, Any]) -> str:
    if not ADD_MODEL_RESIDUE_COMMENTARY:
        return ""
    messages = [
        {"role": "system", "content": "You are a concise Nexus residue commentator. Do not score. Explain unresolved residue in one short paragraph."},
        {"role": "user",   "content": "Prompt:\n" + prompt + "\n\nContract:\n" + json.dumps(contract, ensure_ascii=False, indent=2) + "\n\nAnswer:\n" + answer + "\n\nDeterministic audit:\n" + json.dumps(audit, ensure_ascii=False, indent=2)},
    ]
    return generate_text(messages, max_new_tokens=RESIDUE_MAX_NEW_TOKENS, do_sample=False, use_slot_adapter=False)

print("deterministic critic ready")


deterministic critic ready


In [9]:
# ============================================================
# COMBINED SCORER + KRRB v5 — contract-anchored consensus gate
# ============================================================
def score_candidate_v5(prompt: str, contract: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    answer  = candidate["answer"]
    det     = deterministic_operational_audit(prompt, contract, answer)
    audit   = det["audit"]
    evidence= det["evidence"]

    op         = audit_score(audit)
    forbidden  = evidence["forbidden"]["score"]
    domain     = evidence["domain"]["score"]
    prompt_fit = evidence["prompt_fit"]["score"]

    psi = (
        0.82 * op
        + 0.08 * domain
        + 0.06 * prompt_fit
        + 0.04 * evidence["quality"]
        - 0.10 * max(0.0, forbidden - 0.35)
    )

    support_flags = {
        "F_need":     audit["F_need"]     >= 0.55,
        "F_function": audit["F_function"] >= 0.55,
        "F_boundary": audit["F_boundary"] >= 0.50,
        "F_trap":     audit["F_trap"]     >= 0.55,
        "F_collapse": audit["F_collapse"] >= 0.55,
        "domain":     domain              >= 0.20,
    }
    support = int(sum(1 for v in support_flags.values() if v))

    commentary = ""
    if audit["residue"] and ADD_MODEL_RESIDUE_COMMENTARY:
        commentary = model_residue_commentary(prompt, contract, answer, audit)

    return {
        "branch": candidate["branch"], "psi": float(psi),
        "audit_score": float(op), "support": support,
        "support_flags": support_flags, "audit": audit,
        "evidence": evidence, "residue_commentary": commentary,
        "answer": answer,
    }

def score_all_candidates_v5(prompt: str, contract: Dict[str, Any], candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for cand in candidates:
        print("audit branch:", cand["branch"])
        s = score_candidate_v5(prompt, contract, cand)
        rows.append({
            "branch":       s["branch"],
            "psi":          s["psi"],
            "audit_score":  s["audit_score"],
            "support":      s["support"],
            "F_need":       s["audit"]["F_need"],
            "F_function":   s["audit"]["F_function"],
            "F_boundary":   s["audit"]["F_boundary"],
            "F_trap":       s["audit"]["F_trap"],
            "F_collapse":   s["audit"]["F_collapse"],
            "audit_residue":" | ".join(s["audit"]["residue"]),
            "answer":       s["answer"],
            "detail":       s,
        })
    return pd.DataFrame(rows).sort_values(["psi","support"], ascending=False).reset_index(drop=True)

def answer_agreement(a: str, b: str) -> float:
    wa = wordset(a); wb = wordset(b)
    if not wa or not wb: return 0.0
    return len(wa & wb) / len(wa | wb)

def audit_vector_agreement(row_a: Dict, row_b: Dict) -> float:
    fields = ["F_need","F_function","F_boundary","F_trap","F_collapse"]
    diffs  = [abs(float(row_a[f]) - float(row_b[f])) for f in fields]
    return 1.0 - (sum(diffs) / len(diffs))

def high_quality(row: Dict) -> bool:
    return (
        int(row["support"])      >= SUPPORT_MIN
        and float(row["psi"])          >= PSI_MIN
        and float(row["audit_score"])  >= AUDIT_MIN
    )

def krrb_resolve_v5(
    score_df: pd.DataFrame,
    contract: Optional[Dict[str, Any]],
    contract_ok: bool,
) -> Dict[str, Any]:
    if not contract_ok:
        return {"state": "Ω", "reason": "contract_incomplete_or_unparseable",
                "winner": None, "margin": None, "support": 0}
    if score_df.empty:
        return {"state": "Ω", "reason": "no_candidates",
                "winner": None, "margin": None, "support": 0}

    top    = score_df.iloc[0].to_dict()
    second = score_df.iloc[1].to_dict() if len(score_df) > 1 else None
    second_psi = float(second["psi"]) if second is not None else 0.0
    margin     = float(top["psi"] - second_psi)

    fail = []
    if int(top["support"])     < SUPPORT_MIN: fail.append("support_below_min")
    if float(top["psi"])       < PSI_MIN:     fail.append("psi_below_min")
    if float(top["audit_score"]) < AUDIT_MIN: fail.append("audit_below_min")

    # ── Gate A: normal collapse ──────────────────────────────────────────────
    if not fail and margin >= MARGIN_MIN:
        return {
            "state": "Ψ", "reason": "collapse", "winner": top, "second": second,
            "margin": margin, "support": int(top["support"]), "consensus": False,
        }

    # ── Gate B + C: consensus path ───────────────────────────────────────────
    if not fail and second is not None and margin < MARGIN_MIN:
        both_high        = high_quality(top) and high_quality(second)
        lexical_agr      = answer_agreement(top["answer"], second["answer"])
        audit_agr        = audit_vector_agreement(top, second)
        gate_agreement   = both_high and margin <= CONSENSUS_MARGIN_MAX and (
            lexical_agr >= CONSENSUS_AGREEMENT_MIN or audit_agr >= CONSENSUS_AUDIT_MIN
        )

        if gate_agreement:
            # ── Gate C: contract-anchored stance ────────────────────────────
            stance = contract_stance_agreement(contract, top["answer"], second["answer"])

            if stance["passed"]:
                return {
                    "state": "Ψ", "reason": "consensus_collapse",
                    "winner": top, "second": second,
                    "margin": margin, "support": int(top["support"]),
                    "consensus": True,
                    "agreement": {
                        "lexical": round(lexical_agr, 4),
                        "audit":   round(audit_agr, 4),
                    },
                    "stance": stance,
                }
            else:
                # Gates 1+2 passed but stance disagrees — divergent consensus.
                return {
                    "state": "Ω", "reason": "divergent_consensus",
                    "winner": top, "second": second,
                    "margin": margin, "support": int(top["support"]),
                    "agreement": {
                        "lexical": round(lexical_agr, 4),
                        "audit":   round(audit_agr, 4),
                    },
                    "stance": stance,
                }

        fail.append("margin_below_min_no_consensus")
    elif margin < MARGIN_MIN:
        fail.append("margin_below_min")

    if fail:
        return {
            "state": "Ω", "reason": " | ".join(fail),
            "winner": top, "second": second,
            "margin": margin, "support": int(top["support"]),
        }

    return {
        "state": "Ψ", "reason": "collapse",
        "winner": top, "second": second,
        "margin": margin, "support": int(top["support"]), "consensus": False,
    }

def build_omega_report_v5(
    prompt: str,
    contract_result: Dict[str, Any],
    score_df: pd.DataFrame,
    resolution: Dict[str, Any],
) -> Dict[str, Any]:
    top = resolution.get("winner")
    return {
        "omega_id":            "omega_" + uuid.uuid4().hex[:10],
        "time":                now_iso(),
        "prompt":              prompt,
        "reason":              resolution.get("reason"),
        "contract_parse_error":contract_result.get("parse_error"),
        "contract_complete":   contract_result.get("complete"),
        "contract_original":   contract_result.get("contract_original"),
        "contract":            contract_result.get("contract"),
        "contract_repairs":    contract_result.get("repairs"),
        "top_branch":          None if top is None else top.get("branch"),
        "top_psi":             None if top is None else top.get("psi"),
        "top_audit_score":     None if top is None else top.get("audit_score"),
        "top_support":         resolution.get("support"),
        "margin":              resolution.get("margin"),
        "agreement":           resolution.get("agreement"),
        "stance":              resolution.get("stance"),
        "candidate_scores":    score_df.drop(columns=["detail"]).to_dict(orient="records") if not score_df.empty else [],
    }

print("scorer + KRRB v5 contract-anchored consensus ready")


scorer + KRRB v5 contract-anchored consensus ready


In [10]:
# ============================================================
# LIVE PROMPT
# ============================================================
LIVE_PROMPT = '''
Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.
'''

print(LIVE_PROMPT.strip())


Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.


In [11]:
# ============================================================
# RUN RHI v5
# ============================================================
def run_rhi_v5(prompt: str, save: bool = True) -> Dict[str, Any]:
    run_id = "rhi_v5_" + uuid.uuid4().hex[:10]
    t0     = time.time()

    print("Δ generating + repairing contract...")
    contract_result = generate_contract(prompt)
    contract        = contract_result["contract"]

    print("contract complete:", contract_result["complete"])
    print("contract repairs:", contract_result["repairs"])
    if contract is not None:
        print(json.dumps(contract, ensure_ascii=False, indent=2)[:3000])
    else:
        print("raw contract:"); print(contract_result["raw_contract"])

    if not contract_result["complete"]:
        empty_df   = pd.DataFrame()
        resolution = krrb_resolve_v5(empty_df, contract, contract_ok=False)
        omega      = build_omega_report_v5(prompt, contract_result, empty_df, resolution)
        result     = {
            "run_id": run_id, "time": now_iso(), "prompt": prompt,
            "contract_result": contract_result, "candidates": [],
            "scores": [], "resolution": resolution,
            "answer": None, "omega": omega,
            "elapsed_sec": time.time() - t0,
        }
    else:
        print("Δ generating candidate branches...")
        candidates = generate_candidate_branches(prompt, contract)

        print("Δ deterministic operational audit...")
        score_df   = score_all_candidates_v5(prompt, contract, candidates)
        display(score_df.drop(columns=["detail"]))

        resolution = krrb_resolve_v5(score_df, contract, contract_ok=True)

        if resolution["state"] == "Ψ":
            answer = resolution["winner"]["answer"]
            omega  = None
            reason = resolution["reason"]
            print(f"Ψ {reason}: {resolution['winner']['branch']} margin:{round(resolution['margin'],4)} support:{resolution['support']}")
            if resolution.get("consensus"):
                agr = resolution.get("agreement", {})
                st  = resolution.get("stance", {})
                print(f"  lexical:{agr.get('lexical')}  audit:{agr.get('audit')}  stance:{st.get('aggregate')}")
            print("audit:", round(float(resolution["winner"]["audit_score"]),4),
                  "psi:",  round(float(resolution["winner"]["psi"]),4))
            print()
            print(answer)
        else:
            answer = None
            omega  = build_omega_report_v5(prompt, contract_result, score_df, resolution)
            print("Ω residue:", resolution["reason"])
            if resolution.get("stance"):
                st = resolution["stance"]
                print(f"  stance aggregate: {st['aggregate']} (threshold {CONSENSUS_STANCE_MIN})")
                for field, fd in st["fields"].items():
                    if fd["consistency"] < CONSENSUS_STANCE_MIN:
                        print(f"  field '{field}' failed: ov_a={fd['overlap_a']} ov_b={fd['overlap_b']} consistency={fd['consistency']}")
            print(json.dumps(omega, ensure_ascii=False, indent=2)[:3500])

        result = {
            "run_id": run_id, "time": now_iso(), "prompt": prompt,
            "contract_result": contract_result,
            "candidates": candidates,
            "scores":     score_df.drop(columns=["detail"]).to_dict(orient="records"),
            "resolution": resolution,
            "answer":     answer,
            "omega":      omega,
            "elapsed_sec": time.time() - t0,
        }

    if save:
        out_path = OUTPUT_DIR / "rhi_live_runs_v5.jsonl"
        with out_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False, default=str) + chr(10))
        print(); print("saved:", out_path)

    return result

live_result = run_rhi_v5(LIVE_PROMPT.strip(), save=True)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Δ generating + repairing contract...
contract complete: True
contract repairs: ['removed_generic_domain:current', 'removed_scar:general purpose', 'removed_scar:purpose', 'removed_scar:nonexistent', 'removed_scar:surface label without operational fit', 'removed_scar:wrong neighboring domain carrier', 'removed_scar:string wraps but does not center under rotation', 'removed_scar:permanent membership without local constraint satisfaction', 'removed_scar:name-only rubber-part match', 'added_positive:intent', 'added_positive:boundary', 'added_positive:tool', 'added_positive:agent', 'added_positive:sequence', 'added_positive:tool use after contract', 'added_forbidden:tool before contract', 'added_forbidden:action without boundary', 'added_forbidden:answer guessing', 'added_forbidden:tool-first action', 'rewrote_boundary_polarity']
{
  "family_class": "operational closure of",
  "domain_carrier": [
    "AI",
    "agents",
    "fail",
    "tools",
    "forming",
    "contract",
    "intent",
  

,branch,psi,audit_score,support,F_need,F_function,F_boundary,F_trap,F_collapse,audit_residue,answer
0,operational,0.904276,0.952,6,1.0,0.8,1.0,1.00,1.0,,The failure mode described in the prompt invol...
1,residue_aware,0.866726,0.916,6,1.0,0.8,0.8,1.00,1.0,,The failure in current AI agents using tools b...
2,direct,0.855853,0.919,6,1.0,1.0,0.8,0.75,1.0,,Current AI agents often fail to form contracts...
3,skeptical,0.834038,0.871,6,1.0,0.8,0.8,0.75,1.0,,Current AI agents often fail to form contracts...
4,contract_fit,0.810623,0.826,5,1.0,0.8,0.8,0.50,1.0,,The current AI agents fail when they use tools...


Ψ consensus_collapse: operational margin:0.0376 support:6
  lexical:0.193  audit:0.96  stance:0.916
audit: 0.952 psi: 0.9043

The failure mode described in the prompt involves current AI agents failing when they attempt to use tools before forming a contract. This issue arises because the sequence of actions does not follow the required order: first, the agent needs to form a contract (or intent) with its environment or other entities involved, followed by using tools within this context. The presence of tools before the contract can lead to premature action, which violates the established boundaries and intent of the interaction. This situation often results in the agent either rejecting the initial tool usage due to lack of proper intention or boundary conditions being met, leading to incorrect behavior or even crashing the system.

saved: D:\@User Data\Downloads\rhi_live_runtime_v5_outputs\rhi_live_runs_v5.jsonl


In [12]:
# ============================================================
# BATCH MODE
# ============================================================
PROMPTS = [
    "Why does RAG fail when retrieval happens before intent is stabilized?",
    "Explain LoRA as a groove in a frozen model manifold using Nexus terms.",
    "What does residue repair add to a normal AI agent loop?",
    # divergence stress-test — two operationally correct but differently-stanced answers expected:
    "Is the contract boundary condition a filter or a gate? Defend one operational reading.",
]

RUN_BATCH = False

if RUN_BATCH:
    batch_results = []
    for i, prompt in enumerate(PROMPTS):
        print("=" * 100)
        print("BATCH", i+1, "/", len(PROMPTS))
        batch_results.append(run_rhi_v5(prompt, save=True))
    print("batch complete:", len(batch_results))
else:
    print("batch skipped — set RUN_BATCH = True to run")


batch skipped — set RUN_BATCH = True to run


In [13]:
# ============================================================
# READ SAVED RUNS / MANIFEST
# ============================================================
def read_saved_runs(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

runs_file  = OUTPUT_DIR / "rhi_live_runs_v5.jsonl"
saved_runs = read_saved_runs(runs_file)

summary_rows = []
for r in saved_runs:
    res    = r.get("resolution", {})
    winner = res.get("winner") or {}
    stance = res.get("stance") or {}
    summary_rows.append({
        "run_id":         r.get("run_id"),
        "time":           r.get("time"),
        "state":          res.get("state"),
        "reason":         res.get("reason"),
        "support":        res.get("support"),
        "margin":         res.get("margin"),
        "psi":            winner.get("psi"),
        "audit_score":    winner.get("audit_score"),
        "branch":         winner.get("branch"),
        "stance_agg":     stance.get("aggregate"),
        "stance_passed":  stance.get("passed"),
        "elapsed_sec":    r.get("elapsed_sec"),
        "prompt":         str(r.get("prompt",""))[:180],
    })

runs_df = pd.DataFrame(summary_rows)
display(runs_df)

manifest = {
    "notebook":        "rhi_live_runtime_v5_contract_anchored_consensus",
    "model_name":      MODEL_NAME,
    "slot_adapter_dir":str(SLOT_ADAPTER_DIR),
    "output_dir":      str(OUTPUT_DIR),
    "runs_file":       str(runs_file),
    "n_saved_runs":    len(saved_runs),
    "runtime_shape":   "Q -> C_raw -> C_repaired -> {A_i} -> deterministic five-dimensional audit -> Ψ/Ω",
    "collapse_controls": {
        "support_min":              SUPPORT_MIN,
        "margin_min":               MARGIN_MIN,
        "psi_min":                  PSI_MIN,
        "audit_min":                AUDIT_MIN,
        "consensus_margin_max":     CONSENSUS_MARGIN_MAX,
        "consensus_agreement_min":  CONSENSUS_AGREEMENT_MIN,
        "consensus_audit_min":      CONSENSUS_AUDIT_MIN,
        "consensus_stance_min":     CONSENSUS_STANCE_MIN,
    },
    "v5_gate_sequence": [
        "gate_A: normal collapse — margin >= MARGIN_MIN",
        "gate_B: both high quality + lexical/audit agreement >= threshold",
        "gate_C: contract-anchored stance agreement >= CONSENSUS_STANCE_MIN",
        "Omega_divergent_consensus: B passes, C fails",
    ],
}

(OUTPUT_DIR / "rhi_live_runtime_v5_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8",
)
print(json.dumps(manifest, indent=2))


,run_id,time,state,reason,support,margin,psi,audit_score,branch,stance_agg,stance_passed,elapsed_sec,prompt
0,rhi_v5_2b5a97b7b6,2026-05-04T07:03:29,Ψ,consensus_collapse,6,0.03755,0.904276,0.952,operational,0.916,True,24.000319,"Using the Nexus lens, explain why current AI a..."


{
  "notebook": "rhi_live_runtime_v5_contract_anchored_consensus",
  "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
  "slot_adapter_dir": "D:\\@User Data\\Downloads\\slot_builder_lora_v2",
  "output_dir": "D:\\@User Data\\Downloads\\rhi_live_runtime_v5_outputs",
  "runs_file": "D:\\@User Data\\Downloads\\rhi_live_runtime_v5_outputs\\rhi_live_runs_v5.jsonl",
  "n_saved_runs": 1,
  "runtime_shape": "Q -> C_raw -> C_repaired -> {A_i} -> deterministic five-dimensional audit -> \u03a8/\u03a9",
  "collapse_controls": {
    "support_min": 4,
    "margin_min": 0.04,
    "psi_min": 0.52,
    "audit_min": 0.52,
    "consensus_margin_max": 0.04,
    "consensus_agreement_min": 0.36,
    "consensus_audit_min": 0.86,
    "consensus_stance_min": 0.5
  },
  "v5_gate_sequence": [
    "gate_A: normal collapse \u2014 margin >= MARGIN_MIN",
    "gate_B: both high quality + lexical/audit agreement >= threshold",
    "gate_C: contract-anchored stance agreement >= CONSENSUS_STANCE_MIN",
    "Omega_divergent_con

# Ψ-collapse

v5 fixes the v4 false-consensus risk:

```text
v4:
  both branches high and close in margin
  lexical Jaccard ≥ 0.36 → Ψ(consensus_collapse)
  risk: shared NEXUS vocabulary inflates Jaccard even when stances diverge

v5:
  gate B  — both high + lexical/audit agreement (same as v4)
  gate C  — contract_stance_agreement():
              for each contract field (domain_carrier, preserved_function,
              boundary_conditions, family_class, witness_readout)
              measure how each branch overlaps the field's non-NEXUS terms
              consistency = 1 - |overlap_a - overlap_b|
              aggregate   = mean consistency across fields
              passed      = aggregate ≥ CONSENSUS_STANCE_MIN (0.50)
  if B passes and C fails → Ω_divergent_consensus (labelled, not silently dropped)
```

Diagnostic targets:

```text
True consensus:
  state = Ψ  reason = consensus_collapse
  stance.aggregate ≥ 0.50
  field-level consistency all ≥ 0.50

Divergent consensus (new Ω type):
  state = Ω  reason = divergent_consensus
  stance.aggregate < 0.50
  failing fields printed at runtime
```

Next fold after this run:

```text
compare v4 vs v5 on same prompts
  ↓
identify prompts where v4 gave consensus_collapse but v5 gives divergent_consensus
  ↓
inspect stance.fields to see which contract dimension drove the split
  ↓
those splits are training signal for the repair gate or for a v6 branch merger
```

Collapse chain:

$$\Omega_{\text{margin}} \rightarrow \text{agreement test} \rightarrow \text{stance test} \rightarrow \Psi_{\text{consensus collapse}}$$
